# 03 Figure Export

역할: 저장된 summary artifact에서 보고서/발표용 CSV와 PNG를 재생성한다.
하지 않는 일: runner 실행, runtime setup, exploratory dashboard 표시, 결론 문장 작성.


## Cell Role: Export Path and Helper Setup

저장된 repo를 기준으로 path를 잡고 summary loader만 import한다.
이 노트북은 figure/table 파일을 만드는 용도이므로 interactive dashboard 함수는 import하지 않는다.


In [ ]:
from pathlib import Path
import importlib
import sys

import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
repo_candidates = [cwd, *cwd.parents, Path('/content/ReGraM')]
REPO_ROOT = next((p.resolve() for p in repo_candidates if p.exists() and p.name == 'ReGraM'), cwd)
EXP_ROOT = REPO_ROOT / 'experiments' / 'validation' / 'condition_shift_baseline'
REPORT_ROOT = EXP_ROOT / 'reports'
MANIFEST_ROOT_CANDIDATES = [REPO_ROOT / 'manifests', EXP_ROOT / 'manifests']
SRC_ROOT = EXP_ROOT / 'src'
CORE_SRC = SRC_ROOT / 'core'
for source_root in (SRC_ROOT, CORE_SRC):
    if str(source_root) not in sys.path:
        sys.path.insert(0, str(source_root))

for module_name in ('orchestration.notebook_orchestration', 'orchestration.dashboard_loader'):
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])

from orchestration.dashboard_loader import load_dashboard_frames
from orchestration.notebook_orchestration import (
    SEVERITY_ORDER,
    build_baseline_specs,
    build_requested_run_configs,
    discover_manifest_names,
    display_environment_summary,
    display_title,
    resolve_manifest_paths,
)

RAW_LOCO_ROOT = REPO_ROOT / 'data' / 'row' / 'mvtec_loco_anomaly_detection'
UNIVAD_CAPTION_DATA_ROOT = REPO_ROOT / 'data' / 'mvtec_loco_caption'
BASELINE_SPECS = build_baseline_specs(
    repo_root=REPO_ROOT,
    exp_root=EXP_ROOT,
    raw_loco_root=RAW_LOCO_ROOT,
    univad_caption_data_root=UNIVAD_CAPTION_DATA_ROOT,
    default_patchcore_device='cuda',
)
display_environment_summary(REPO_ROOT, EXP_ROOT, REPORT_ROOT)


## Cell Role: Load Export Source Frames

export 대상 baseline/category/manifest를 지정하고 summary JSON을 DataFrame으로 로드한다.
보고서 figure는 이 셀에서 로드된 `dashboard_frames`만 사용한다.


In [ ]:
DASHBOARD_BASELINES = ['UniVAD']
CATEGORIES = ['breakfast_box']
CONFIGURED_MANIFEST_NAMES = ['query_motion_blur.jsonl', 'query_low_light.jsonl', 'query_gaussian_noise.jsonl']
SELECTED_SEVERITIES = []  # Must match 01; [] means all. Examples: ['low'], ['medium'], ['high']
AUTO_DISCOVER_MANIFESTS = False
EXCLUDED_MANIFEST_NAMES = {'query_identity.jsonl', 'query_multi.jsonl'}
WANDB_PROJECT = 'regram-condition-shift'
WANDB_MODE = 'online'
TOP_K_WORST_SHIFTS = 10

ACTIVE_MANIFEST_NAMES = discover_manifest_names(
    manifest_roots=MANIFEST_ROOT_CANDIDATES,
    configured_names=CONFIGURED_MANIFEST_NAMES,
    auto_discover=AUTO_DISCOVER_MANIFESTS,
    excluded_names=EXCLUDED_MANIFEST_NAMES,
)
manifest_paths = resolve_manifest_paths(ACTIVE_MANIFEST_NAMES, MANIFEST_ROOT_CANDIDATES)
summary_sources = build_requested_run_configs(
    active_baselines=DASHBOARD_BASELINES,
    specs=BASELINE_SPECS,
    categories=CATEGORIES,
    manifest_paths=manifest_paths,
    manifest_names=ACTIVE_MANIFEST_NAMES,
    report_root=REPORT_ROOT,
    wandb_project=WANDB_PROJECT,
    wandb_mode=WANDB_MODE,
    selected_severities=SELECTED_SEVERITIES,
)
dashboard_frames = load_dashboard_frames(
    summary_sources=summary_sources,
    dashboard_baselines=DASHBOARD_BASELINES,
    categories=CATEGORIES,
    report_root=REPORT_ROOT,
    severity_order=SEVERITY_ORDER,
    top_k_worst_shifts=TOP_K_WORST_SHIFTS,
)
display_title('Loaded Summary Sources')
display(pd.DataFrame([
    {
        'baseline': config['baseline'],
        'category': config['category'],
        'summary_exists': config['summary_path'].exists(),
        'severities': ', '.join(config.get('selected_severities', [])) or 'all',
        'summary_path': str(config['summary_path']),
    }
    for config in summary_sources
]))


## Cell Role: Export Tables and Heatmaps

`reports/figures/tables/*.csv`와 `reports/figures/*.png`를 재생성한다.
새 runner 실행 없이 저장된 summary를 재가공하는 단계다.


In [ ]:
import re

import matplotlib.pyplot as plt
import numpy as np

FIGURE_ROOT = REPORT_ROOT / 'figures'
TABLE_ROOT = FIGURE_ROOT / 'tables'
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
TABLE_ROOT.mkdir(parents=True, exist_ok=True)

def slug(value: object) -> str:
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(value)).strip('_')

for key, df in dashboard_frames.items():
    if isinstance(df, pd.DataFrame) and not df.empty:
        df.to_csv(TABLE_ROOT / f'{key}.csv', index=False)

shift_df = dashboard_frames['shift_df']
if not shift_df.empty:
    for metric_name, title, cmap in [
        ('shifted_normal_fpr', 'Shifted-normal FPR', 'YlOrRd'),
        ('image_auroc_drop_from_clean', 'AUROC drop from clean', 'YlOrBr'),
        ('image_auroc_drop_from_clean_logical', 'Logical AUROC drop from clean', 'YlOrBr'),
        ('image_auroc_drop_from_clean_structural', 'Structural AUROC drop from clean', 'YlOrBr'),
        ('median_score_shift', 'Median score shift', 'PuBuGn'),
    ]:
        for (baseline, category), plot_df in shift_df.groupby(['baseline', 'category']):
            pivot = plot_df.pivot_table(index='shift_family', columns='severity', values=metric_name, aggfunc='mean')
            ordered_cols = [column for column in ['low', 'medium', 'high'] if column in pivot.columns]
            pivot = pivot[ordered_cols]
            fig, ax = plt.subplots(figsize=(max(5, 1.4 * len(ordered_cols)), max(3.5, 0.55 * len(pivot))), constrained_layout=True)
            im = ax.imshow(pivot.to_numpy(dtype=float), aspect='auto', cmap=cmap)
            ax.set_title(f'{baseline} | {category} | {title}')
            ax.set_xticks(np.arange(len(pivot.columns)))
            ax.set_xticklabels(pivot.columns)
            ax.set_yticks(np.arange(len(pivot.index)))
            ax.set_yticklabels(pivot.index)
            for row_idx, shift_family in enumerate(pivot.index):
                for col_idx, severity in enumerate(pivot.columns):
                    value = pivot.loc[shift_family, severity]
                    if pd.notna(value):
                        ax.text(col_idx, row_idx, f'{value:.2f}', ha='center', va='center', fontsize=8)
            fig.colorbar(im, ax=ax, shrink=0.8)
            output_path = FIGURE_ROOT / f'{slug(baseline)}_{slug(category)}_{slug(metric_name)}_heatmap.png'
            fig.savefig(output_path, dpi=180)
            plt.close(fig)

display_title('Exported Figure Artifacts')
display(pd.DataFrame([
    {'path': str(path), 'bytes': path.stat().st_size}
    for path in sorted(FIGURE_ROOT.rglob('*'))
    if path.is_file()
]))
